# Map of Italian Science — Country-Level Citation Analysis

## Research Questions
This notebook addresses the country-level component of the *Map of Italian Science* study.

1. **(RQ1 – Map of Italian Science)** What are the institutions and countries that either cites or are cited by the IRIS publications included in OpenCitations of a given institution?
2. **(RQ1a)** Are different institutions showing different citation patterns, depending on the specific case?

The analysis covers six institutions:

- UNIBO (University of Bologna)
- UNIMI (University of Milan)
- UNIPD (University of Padua)
- UNITO (University of Turin)
- UPO (University of Eastern Piedmont)
- SNS (Scuola Normale Superiore)

## Data

For each institution, two country-level datasets are available:

- `citation_counts_countries_incoming.csv`
- `citation_counts_countries_outgoing.csv`

Each dataset contains:

| Column | Description |
|----------|----------|
| country_code | ISO-3166 alpha-2 country code |
| country_name | Country name |
| count | Number of citation relationships |

---
> **Notebook structure**
> 1. Setup & data loading
> 2. Cross-institution comparison (all 6 institutions)
>    - 3a. Top-N countries per institution
>    - 3b. Asymmetry analysis
>    - 3c. Shared vs unique citation partners
>    - 3d. Heatmap: institutions × countries
> 3. Summary of findings


## 1. Setup & Data Loading

The analysis relies on Pandas for data manipulation and Plotly for interactive visualization.

Data-loading, cleaning, and validation routines have been modularized into reusable functions located in the `src/` directory. This notebook therefore focuses exclusively on exploratory analysis and visualization.

In [41]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pycountry
from plotly.subplots import make_subplots
from pathlib import Path
import sys

# ── Paths & Environment ──
try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path.cwd()

# Add parent directory of data_viz to sys.path to allow importing from src
if CURRENT_DIR.name == "data_viz":
    sys.path.append(str(CURRENT_DIR.parent))
else:
    sys.path.append(str(CURRENT_DIR))

from src.data_utils import * 

from src.validation import * 

# ── Visualization settings ──

# ── Colour palette (one colour per institution, consistent across all plots) ──
INST_COLORS = {
    "UNIBO": "#264653",
    "UNIMI": "#2a9d8f",
    "UNIPD": "#8ab17d",
    "UNITO": "#e9c46a",
    "UPO":   "#f4a261",
    "SNS":   "#e76f51",
}

# ── Direction colours ──
DIR_COLORS = {"incoming": "#B7990D", "outgoing": "#320E3B"}

### Loading the Analysis Dataset

Country names are normalized during loading to resolve naming inconsistencies (e.g., *Russia* vs. *Russian Federation*). Citation counts associated with equivalent countries are aggregated automatically.

The resulting dataframe contains all institutions and both citation directions in a unified long-format structure.

In [42]:
# ── Load everything ──
# Long-format, Italy excluded, all institutions
all_df = load_all(exclude_self=True)

print(f"Total rows: {len(all_df):,}")
print(f"Institutions: {all_df['institution'].unique()}")
print(f"Directions:   {all_df['direction'].unique()}")
all_df.head()


Total rows: 2,591
Institutions: ['UNIBO' 'UNIMI' 'UNIPD' 'UNITO' 'UPO' 'SNS']
Directions:   ['incoming' 'outgoing']


,country_code,country_name,count,institution,direction
0,AD,Andorra,19,UNIBO,incoming
1,AE,United Arab Emirates,19714,UNIBO,incoming
2,AF,Afghanistan,432,UNIBO,incoming
3,AG,Antigua and Barbuda,1952,UNIBO,incoming
4,AL,Albania,1403,UNIBO,incoming


### Optional Data Validation

The following utilities are not required for the analysis itself. They are provided for quality-control purposes and should only be executed when new datasets are received or when country normalization rules are updated.

- `validate_dataset()` checks for duplicate country codes and missing values.
- `export_cleaned_csvs()` generates cleaned versions of the original datasets after normalization and aggregation.

In [43]:
# for inst in INSTITUTIONS:
#     for direction in ["incoming", "outgoing"]:
#         print(f"\n{inst} {direction}")
#         validate_dataset(
#             load_country_data(inst, direction),
#             duplicate_subset=["country_code"]
#         )

# export_cleaned_csvs(
#     loader_function=load_country_data,
#     filename_prefix="countries",     
#     output_dir=BASE_PATH.parent / "visualizations",
#     institutions=INSTITUTIONS
# )

## Asymmetry map

In [44]:
import plotly.graph_objects as go

# Build a wide dataframe for each institution and store traces
frames = {}
for inst in INSTITUTIONS:
    df = load_institution(inst, exclude_self=True)
    wide = pivot_directions(df)
    wide["country_iso3"] = wide["country_code"].apply(to_iso3)
    wide = wide.dropna(subset=["country_iso3"])
    frames[inst] = wide

# Build one choropleth trace per institution

custom_colorscale = [
    [0.0,  "#FFD500"],  # strong inbound bias
    [0.25, "#FFE760"],  # mild inbound
    [0.5,  "#F5F0E8"],  # balanced
    [0.75, "#6B3E7A"],  # mild outbound
    [1.0,  "#23022E"],  # strong outbound bias
]

traces = []
for inst in INSTITUTIONS:
    wide = frames[inst]
    traces.append(go.Choropleth(
        locations=wide["country_iso3"],
        z=wide["log2_ratio"],
        text=wide["country_name"],
        customdata=np.stack([
            wide["incoming_count"],
            wide["outgoing_count"],
            wide["log2_ratio"]
        ], axis=1),
        hovertemplate=(
            "<b>%{text}</b><br>" +
            "Incoming: %{customdata[0]:,.0f}<br>" +
            "Outgoing: %{customdata[1]:,.0f}<br>" +
            "log₂(out/in): %{customdata[2]:.2f}<extra></extra>"
        ),
        colorscale=custom_colorscale,
        zmid=0,
        zmin=-3,
        zmax=3,
        colorbar=dict(title="log₂(out/in)"),
        visible=(inst == "UNIBO")  # only UNIBO visible initially
    ))

# Build dropdown buttons — each button makes one trace visible
buttons = []
for i, inst in enumerate(INSTITUTIONS):
    visibility = [inst == other for other in INSTITUTIONS]  # True only for selected
    buttons.append(dict(
        label=INSTITUTION_LABELS[inst],
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"Citation Asymmetry — {INSTITUTION_LABELS[inst]} (log₂ outgoing/incoming)"}
        ]
    ))

fig = go.Figure(data=traces)

fig.update_layout(
    title=f"Citation Asymmetry — {INSTITUTION_LABELS['UNIBO']} (log₂ outgoing/incoming)",
    title_x=0.5,
    template="plotly_white",
    height=500,
    geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction="down",
        x=0.02,
        xanchor="left",
        y=1.1,
        yanchor="top",
        showactive=True,
        bgcolor="white",
        bordercolor="gray",
        font=dict(size=12)
    )],
    annotations=[dict(
        text="Institution:",
        x=0.02, y=1.15,
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=12)
    )]
)

fig.show()

---

### 3. Cross-Institution Comparison

Now we scale the same analyses to all six institutions to answer **RQ1a/RQ1b**: do institutions show different citation patterns?



### 3a. Proportional stacked bar chart

This is the starting point for investigating the international connections of the six Italian institutions. First, we establish the baseline geographic distribution of their citation networks. By calculating the proportional contribution of the top 10 cited and citing countries, we map the fundamental orientation of these institutions within the global scientific landscape.

This step is essential to verify whether the institutions operate within a shared international framework or if there are immediate, macro-level discrepancies in their reach.

In [45]:

def plot_proportional_citations(direction, title):
    all_data = []
    
    for inst in INSTITUTIONS:
        df = load_country_data(inst, direction)
        # df = df[df["country_code"] != "IT"]
        all_data.append(df)   
        
    combined_df = pd.concat(all_data, ignore_index=True)
    
    top_10_countries = combined_df.groupby('country_name')['count'].sum().nlargest(10).index

    # Filter the dataframe to keep only those 10 countries
    final_df = combined_df[combined_df['country_name'].isin(top_10_countries)].copy()
    totals = final_df.groupby('institution')['count'].transform('sum')
    final_df['percentage'] = (final_df['count'] / totals) * 100

    colors_10 = [
        '#320E3B',
        '#4D1343',
        '#69184B',
        '#861D53',
        '#A3225B',
        '#BF2862',
        '#C84457',
        '#D0604D',
        '#C47D30',
        '#B7990D'
    ]

    # Create the Plotly chart
    fig = px.bar(
        final_df,
        x="institution",
        y="percentage",
        color="country_name",
        title=title,
        category_orders={"country_name": list(top_10_countries)}, 
        color_discrete_sequence=colors_10,
        labels={
            "Percentage": "Proportion among Top 10 (%)",
            "country_name": "Country",
            "Institution": "Institution"
        },
        hover_data={"count": True, "percentage": ':.2f'} 
    )

    fig.update_layout(
        barmode='stack',
        template="plotly_white",
        title_font=dict(size=18, family="Arial, sans-serif"),
        hoverlabel=dict(bgcolor="white", font_size=13),
        legend=dict(
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02
        )
    )

    fig.show()

# Citing entities (incoming)
plot_proportional_citations(
    direction='incoming', 
    title='International Geographic Distribution of Citing Entities (Top 10)'
)

# Cited entities (outgoing)
plot_proportional_citations(
    direction='outgoing', 
    title='International Geographic Distribution of Cited Entities (Top 10)'
)

### Results

At a macro level, the geographic distribution of both inbound (cited) and outbound (citing) citations reveals a highly isomorphic pattern across the six Italian universities. The stacked proportional bar charts confirm that these institutions are firmly integrated into the traditional global scientific core.

Specifically, the US, France and the UK consistently emerge as the dominant hubs, collectively accounting for the vast majority of international citation flows.

However, while these charts demonstrate a shared macro-level integration, they also mask the specific insitutional behavior hidden beneath these uniform volume aggregates. To further analyze the data and deepen our exploration we use alternatives highlighting how each insitution differs from the global network and other hidden patterns.


### 3b. Inbound vs. Outbound Citation Partners — Diverging Bar Charts

For each of the six institutions, the chart below displays the top 12 countries by total citation volume (inbound + outbound combined, Italy excluded), arranged as a diverging horizontal bar chart. Gold bars extend leftward and represent **inbound** citations — publications from that country that cite the institution's IRIS output indexed in OpenCitations. Dark purple bars extend rightward and represent **outbound** citations — references made by the institution's publications to works from that country.

Countries are sorted by total volume (most cited partners at the top). Each subplot has **an independent x-axis**, scaled symmetrically around zero to the maximum value observed for that institution. This choice makes asymmetries readable regardless of institutional size: the University of Bologna and University of Milan operate on a ±5M scale, while UPO and SNS operate on a ±1–2M scale.


In [46]:
TOP_N = 12  # per institution; slightly more than before since both directions share the same bars

from plotly.subplots import make_subplots
import plotly.graph_objects as go

INST_ORDER = ["UNIBO", "UNIMI", "UNIPD", "UNITO", "UPO", "SNS"]
NROWS, NCOLS = 2, 3

fig = make_subplots(
    rows=NROWS, cols=NCOLS,
    subplot_titles=[INSTITUTION_LABELS[i] for i in INST_ORDER],
    shared_xaxes=False,
    shared_yaxes=False,
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

for idx, inst in enumerate(INST_ORDER):
    row = idx // NCOLS + 1
    col = idx % NCOLS + 1

    df = load_institution(inst, exclude_self=True)
    wide = pivot_directions(df)
    top = wide.head(TOP_N).copy()

    # Sort ascending so the longest bar is at the top in the plot
    top = top.sort_values("total", ascending=True)

    color = INST_COLORS[inst]

    # incoming bars go LEFT (negative x)
    fig.add_trace(go.Bar(
        x=-top["incoming_count"],
        y=top["country_name"],
        orientation="h",
        name="incoming",
        marker_color=DIR_COLORS["incoming"],
        showlegend=(idx == 0),
        legendgroup="incoming",
        hovertemplate="<b>%{y}</b><br>incoming: %{customdata:,}<extra></extra>",
        customdata=top["incoming_count"],
    ), row=row, col=col)

    # outgoing bars go RIGHT (positive x)
    fig.add_trace(go.Bar(
        x=top["outgoing_count"],
        y=top["country_name"],
        orientation="h",
        name="outgoing",
        marker_color=DIR_COLORS["outgoing"],
        showlegend=(idx == 0),
        legendgroup="outgoing",
        hovertemplate="<b>%{y}</b><br>outgoing: %{customdata:,}<extra></extra>",
        customdata=top["outgoing_count"],
    ), row=row, col=col)

    # Zero line per subplot
    fig.add_vline(x=0, line_width=1, line_color="gray", row=row, col=col)

fig.update_layout(
    title=f"Incoming vs outgoing Citation Partners — Top {TOP_N} Countries per Institution (excl. Italy)",
    title_x=0.5,
    barmode="overlay",
    template="plotly_white",
    height=850,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="center", x=0.5,
        title_text=""
    )
)

# Symmerise x-axes so zero is centred — compute per institution
for idx, inst in enumerate(INST_ORDER):
    df = load_institution(inst, exclude_self=True)
    wide = pivot_directions(df)
    top = wide.head(TOP_N)
    max_val = max(top["incoming_count"].max(), top["outgoing_count"].max()) * 1.1
    xaxis_key = "xaxis" if idx == 0 else f"xaxis{idx+1}"
    fig.layout[xaxis_key].update(range=[-max_val, max_val])

fig.show()

### Findings

**Shared geographic structure.** The citation geography is strikingly consistent across all six institutions. The United States, France, the United Kingdom, Germany, China, and Spain appear in every institution's top 12 in both directions, suggesting that these partnerships reflect the broader structure of international science rather than institution-specific strategies.

**Systematic outbound dominance.** Across all institutions, outbound bars are consistently longer than inbound bars — particularly for the United States. Italian universities collectively cite American science far more than American science cites them back. This pattern is well-documented in the scientometrics literature and reflects both the volume of US publication output and citation practices within disciplines where US journals dominate.

**China asymmetry.** China is a notable exception to the outbound-dominant pattern. For most institutions, the inbound bar for China is visibly longer than or comparable to the outbound bar, meaning Chinese publications cite Italian research more than Italian publications cite Chinese research. This likely reflects a combination of Western-centric citation practices and the disciplinary composition of each institution's output.

**France as the most balanced partner.** France consistently shows the most symmetric bars across institutions — a sign of genuine bilateral exchange rather than a directional dependency. This likely reflects shared European research infrastructures and co-authorship networks.

**Institution-specific partners.** Below the top five countries, some divergence appears. **Russia** appears in the top 12 for SNS, UPO and UNIBO but not for the other three universities. Same for **India** which enters the top 12 for UPO, UNITO, SNS, and UNIPD but not for UNIBO or UNIMI. These divergences across otherwise similar large institutions is worth noting and may reflect disciplinary composition differences. 
**South Korea** and **Turkey** appear for SNS and UPO. These differences likely reflect disciplinary specialisations: SNS's concentration in mathematics and physics, and UPO's smaller size making niche international partnerships more visible in relative terms.

**Scale differences are meaningful.** The raw volume difference between UNIBO/UNIMI/UNIPD/UNITO (±5M range) and UPO/SNS (±1–2M range) reflects differences in institutional size and publication output rather than citation behaviour per se. Comparisons of *shape* (which countries appear, how symmetric the bars are) are more informative across institutions than comparisons of raw counts.

## 3c. Cross-Institutional Comparison - Relative Specialization Heatmaps

As stated before, the stacked bar charts confirm broad integration into global networks, but they mask the institution-specific specializations that differentiate the research profiles of the six Italian institutions.

Moving beyond raw volume, we compare all six institutions using a Relative Specialization Heatmap. By calculating each institution's deviation from the group mean (in percentage points), we mathematically subtract the macro-level dominance of the US/EU core.

The resulting color matrix isolates unique institutional "fingerprints" - showing exactly where a specific university over-indexes (dark purple) or under-indexes (dark gold) geographically compared to its Italian peers.

In [69]:
def plot_heatmap(direction, title, top_n=15):
    all_data = []

    for inst in INSTITUTIONS:
        # Use the shared loader with cleaning pipeline
        df = load_country_data(inst, direction)
        df = df[df["country_code"] != "IT"]
        all_data.append(df)

    combined_df = pd.concat(all_data, ignore_index=True)

    # Calculate true proportions before filtering countries
    totals = combined_df.groupby("institution")["count"].transform("sum")
    combined_df["Proportion"] = (combined_df["count"] / totals) * 100

    # Identify Top N countries overall
    top_countries = (combined_df.groupby("country_name")["count"]
                     .sum()
                     .nlargest(top_n)
                     .index)

    # Filter data and pivot
    heatmap_df = combined_df[combined_df["country_name"].isin(top_countries)].copy()
    pivot_df = (heatmap_df.pivot(index="country_name", columns="institution", values="Proportion")
                .fillna(0))
    pivot_df = pivot_df[INSTITUTIONS]

    # Calculate Average and Deviation
    pivot_df["Average"] = pivot_df.mean(axis=1)
    deviation_df = pivot_df[INSTITUTIONS].sub(pivot_df["Average"], axis=0)

    deviation_df["Average"] = pivot_df["Average"]
    deviation_df = deviation_df.sort_values(by="Average", ascending=True)

    averages = deviation_df["Average"].values
    countries = deviation_df.index.tolist()

    y_labels_with_avg = [f"{country} (Avg: {avg:.1f}%)"
                         for country, avg in zip(countries, averages)]

    true_proportions = deviation_df[INSTITUTIONS].values + averages[:, None]

    deviation_df = deviation_df.drop(columns=["Average"])

    custom_colorscale = [
        [0.0, "#E05D53"],  # Under-indexing
        [0.5, "#F4F4F6"],  # Average
        [1.0, "#320E3B"]   # Over-indexing
    ]

    vmax = np.abs(deviation_df.values).max()

    fig = go.Figure(data=go.Heatmap(
        z=np.round(deviation_df.values, 2),  
        x=deviation_df.columns,
        y=y_labels_with_avg,
        customdata=true_proportions[..., np.newaxis],
        colorscale=custom_colorscale,
        zmin=-vmax,
        zmax=vmax,
        zmid=0,
        hovertemplate=(
            "<b>Institution:</b> %{x}<br>" +
            "<b>Country:</b> %{y}<br>" +
            "<b>Deviation from Avg:</b> %{z:+.1f} pp<br>" +
            "<b>True Proportion:</b> %{customdata[0]:.1f}%<br>" +
            "<extra></extra>"
        ),
        colorbar=dict(
            title="Deviation (pp)",
            title_side="right",
            tickvals=[-2, -1, 0, 1, 2],
            ticktext=["−2", "−1", "0", "+1", "+2"],
            tickfont=dict(size=11)
            ),
        xgap=2,
        ygap=2
    ))

    fig.update_layout(
        title=dict(text=title, font=dict(size=18, family="Arial, sans-serif")),
        template="plotly_white",
        xaxis=dict(title="", tickfont=dict(size=13, weight="bold", family="Arial, sans-serif"), side="top", tickangle=0),
        yaxis=dict(title="", tickfont=dict(size=13, family="Arial, sans-serif"), ticklabelposition="outside right"),
        width=850,
        height=600,
        margin=dict(t=120, l=180)
    )

    annotations = []
    for i, row in enumerate(deviation_df.values):
        for j, val in enumerate(row):
            text_color = "white" if abs(val) > (vmax * 0.4) else "black"
            annotations.append(dict(
                x=deviation_df.columns[j],
                y=y_labels_with_avg[i],
                text=f"{val:+.1f}",
                font=dict(color=text_color, size=10),
                showarrow=False
            ))

    # Add subtitle explaining how to read the chart
    annotations.append(dict(
        x=0.5,
        y=1.06,
        xref="paper",
        yref="paper",
        text="Each cell shows percentage points above (+) or below (−) the six-institution average. White = at the average.",
        showarrow=False,
        font=dict(size=11, color="gray", family="Arial, sans-serif"),
        xanchor="center",
        yanchor="bottom"
    ))

    fig.update_layout(annotations=annotations)
    fig.show() 


# incoming
plot_heatmap(
    direction="incoming",
    title="Relative Geographic Specialization: Incoming Citations (Top 15 Countries)"
)

# outgoing
plot_heatmap(
    direction="outgoing",
    title="Relative Geographic Specialization: Outgoing Citations (Top 15 Countries)"
)

### Results

By calculating each university's deviation from the group mean, the relative specialization heatmaps reveal distinct institutional behaviors that are invisible in raw volume counts.

For instance, the Scuola Normale Superiore (SNS) shows a significantly higher reliance on the US - since the resulting deviation is +2.9pp from the average. Conversely, institutions such as UPO and UNITO exhibit a deeper structural alignment with Chinese research networks.

These deviations indicate that while the Italian science core operates within a unified global hierarchy, individual geographic spheres of influence might be dictated by other aspects, such as localized strategic priorities and specific disciplinary focuses.

### 3d. Asymmetry Comparison Across Institutions

In [48]:
# For each institution compute the mean log2_ratio across its top countries
# as a summary measure of net citation balance

summary_rows = []
for inst in INSTITUTIONS:
    df = load_institution(inst, exclude_self=True)
    wide = pivot_directions(df)
    top20 = wide.head(20)
    summary_rows.append({
        "institution": inst,
        "label": INSTITUTION_LABELS[inst],
        "mean_log2_ratio": top20["log2_ratio"].mean(),
        "n_incoming_countries": (df[df["direction"]=="incoming"]["count"] > 0).sum(),
        "n_outgoing_countries": (df[df["direction"]=="outgoing"]["count"] > 0).sum(),
        "total_incoming": df[df["direction"]=="incoming"]["count"].sum(),
        "total_outgoing": df[df["direction"]=="outgoing"]["count"].sum(),
    })

summary = pd.DataFrame(summary_rows)
summary["total"] = summary["total_incoming"] + summary["total_outgoing"]
summary["pct_outgoing"] = summary["total_outgoing"] / summary["total"] * 100
summary.set_index("institution", inplace=True)
summary


,label,mean_log2_ratio,n_incoming_countries,n_outgoing_countries,total_incoming,total_outgoing,total,pct_outgoing
institution,,,,,,,,
UNIBO,University of Bologna,-0.076902,226,225,29872589,30124137,59996726,50.209635
UNIMI,University of Milan,-0.402483,227,222,32727946,27257280,59985226,45.439989
UNIPD,University of Padua,-0.165805,228,223,28571838,26963079,55534917,48.551579
UNITO,University of Turin,-0.030687,225,219,18823947,19447727,38271674,50.814937
UPO,University of Eastern Piedmont,0.032534,213,208,6401677,6717834,13119511,51.204912
SNS,Scuola Normale Superiore,0.006932,194,181,6455250,6531698,12986948,50.294326


In [49]:
fig = px.bar(
    summary.reset_index(),
    x="institution",
    y="mean_log2_ratio",
    color="institution",
    color_discrete_map=INST_COLORS,
    title="Mean Citation Asymmetry per Institution (log₂ outgoing/incoming, top 20 countries)",
    labels={"mean_log2_ratio": "Mean log₂(outgoing/incoming)",
            "institution": ""},
    template="plotly_white",
    text_auto=".2f"
)
fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray",
              annotation_text="balanced", annotation_position="right")
fig.update_layout(title_x=0.5, showlegend=False, height=420)
fig.show()


### Interpretation

- Positive values (outbound-biased): the institution acts as a net "exporter" of citations, indicating that its reliance on international literature exceeds the incoming recognition it receives from abroad.

- Negative Values (Inbound-Biased): The institution acts as a net "importer" of citations, signaling that its research output is more frequently referenced by international peers than it references them in return.

### Results

The citation asymmetry profile reveals a distinct trend among the major research-intensive universities. UNIBO, UNIMI, UNIPD, and UNITO all exhibit negative $\log_2$ values, confirming their status as net citation "importers" within their top 20 international partners. The data indicates that these institutions are cited by the global community significantly more often than they cite back.
Among these, **UNIMI** demonstrates the most pronounced inbound bias (-0.40). Conversely, the smaller institutions (**UPO** and **SNS**) display values approaching zero, suggesting a more balanced, reciprocal citation exchange with their primary international partners.

This divergence highlights a structural difference in how research-intensive universities engage with global knowledge production compared to smaller, more specialized institutions.

### 3e. Volume vs. Diversity Scatter

Finally, we evaluate the structural breadth of these citation networks. This scatter plot compares total citation volume (x-axis, log-scaled) against the number of distinct country partners (y-axis). 

This allows us to quickly assess whether an institution's global influence is highly concentrated in a few key nations (lower on the y-axis) or widely distributed across a diverse, highly internationalized network (higher on the y-axis).

In [50]:
# A scatter where:
#   x = total citation volume (log scale)
#   y = number of distinct countries with at least 1 citation
#   shape = direction

scatter_rows = []
for inst in INSTITUTIONS:
    df = load_institution(inst, exclude_self=True)
    for direction in ["incoming", "outgoing"]:
        sub = df[df["direction"] == direction]
        scatter_rows.append({
            "institution": inst,
            "label": INSTITUTION_LABELS[inst],
            "direction": direction,
            "total": sub["count"].sum(),
            "n_countries": (sub["count"] > 0).sum(),
        })

scatter_df = pd.DataFrame(scatter_rows)

fig = px.scatter(
    scatter_df,
    x="total", y="n_countries",
    color="institution",
    symbol="direction",
    text="institution",
    color_discrete_map=INST_COLORS,
    log_x=True,
    title="Citation Volume vs. Geographic Diversity by Institution & Direction",
    labels={"total": "Total citations (log scale)",
            "n_countries": "Number of distinct countries"},
    template="plotly_white",
    height=480
)
fig.update_traces(textposition="top center", marker_size=11)
fig.update_layout(title_x=0.5)
fig.show()


### Interpretation

Each institution appears twice (circle = inbound, triangle = outbound). Institutions in the top-right corner have both high volume and broad geographic spread. Institutions in the bottom-left are small and geographically concentrated. Divergence between the two symbols for the same institution indicates that one direction is more globally distributed than the other.

### Results

The Scatter plot maps the relationship between total citation volume and the geographic breadth of the citation network, yielding two key insights:
- **Institutional Clustering**: The data displays a clear bifurcation in network architecture. The research-intensive universities (UNIBO, UNIMI, UNIPD, UNITO) form a high-volume, high-diversity cluster in the top-right quadrant, indicating both extensive citation traffic and a widely distributed international footprint. In contrast, UPO and SNS occupy the lower-left, reflecting significantly lower absolute volumes and more geographically concentrated citation networks.
- **Directional Divergence**: Across all six institutions, the inbound citation markers (circles) consistently outperform the outbound markers (diamonds) in both total volume and geographic reach. This systematic gap reinforces the "net receiver" narrative identified in the previous asymmetry analysis; these Italian institutions are not only cited more often than they cite, but their "inbound" intellectual influence is geographically broader than the scope of the literature they cite in return.

---

## 4. Summary of Findings: Cross-Institutional Synthesis

### 4.1 Common patterns across all six Italian institutions
The empirical data confirms that Italian academic influence is anchored by a stable, recurring core of international partners.
- The **United Kingdom, United States, Germany, and France** consistently appear in the top-10 citation partners across all six institutions, serving as the primary anchor points for international epistemic exchange.
- While US-centricity is universal, the scale of this dependency varies significantly: larger, research-intensive universities (UNIBO, UNIMI, UNIPD, UNITO) exhibit a more intense, high-volume reliance on American output compared to smaller, specialized institutions, suggesting that the "US effect" scales in proportion to institutional output.


### 4.2 Institution-specific patterns (RQ1a — inbound / RQ1b — outbound)
The analysis reveals a clear bifurcation in network architecture based on institutional mission:
- **Specialization Effects**: SNS displays a distinct, more European-focused profile, reflecting its specific humanities and social science focus which tends to favor localized or regional co-authorship networks.
- **Network Breadth vs. Scale**: UPO exhibits a more concentrated and less globally distributed citation network, contrasting with the wide geographic footprint of the multidisciplinary giants.
- **Institutional Convergence**: The larger universities (UNIBO, UNIMI, UNIPD, UNITO) exhibit high convergence, sharing a cohesive "Italian model" of international engagement. This model is characterized by extensive, diversified global networks that transcend individual disciplinary strengths.


### 4.3 Structural Asymmetry & Dependencies
The findings reframe Italian institutions as net "importers" of knowledge. When comparing aggregate inbound vs. outbound citation volumes, these universities consistently demonstrate an inbound bias, receiving more citations from the international community than they generate in return.

A notable structural anomaly persists regarding **China**: there is a consistent "one-way" citation pattern across all institutions, where Italian research receives significant inbound citations from Chinese sources, while citing Chinese scholarship at substantially lower relative rates.


### 4.4 Methodological Considerations
This analysis relies on a rigorous methodological framework to ensure validity:
- **Excluding Italy** from the analysis: exclusion of Italian-to-Italian citations was essential to mitigate domestic noise and isolate pure international relationships. Because the data is already highly right-skewed - with countries like the US dominating the charts - leaving Italy in the dataset would compress the rest of the world into an unreadable "long tail”. Furthermore, exlcuding Italy, we filter out the localized bias - researchers citing their own past work and local academic bubbles - and reach a purer metric of how these institutions are perceived and utilized on the global stage.
- **Scaling Strategies**: We utilize $log_{10}$ scaling for geographic map visualizations to manage extreme distribution skew, while retaining linear axes for ranked bar charts to preserve the clarity of high-volume partners.
- **Asymmetry Metric**: The $log_{2}$ ratio provides a neutral, interpretable metric of directional bias. A value of $log_{2} = 1$ indicates a 2:1 citation ratio, serving as a reliable index for institutional asymmetry.Volume vs. Pattern: When comparing smaller institutions (SNS, UPO) to larger ones, the analysis prioritizes citation patterns (ranks and relative ratios) over absolute volumes to account for inherent differences in institutional size and output.